# Persian Question Answering - ParsBERT Fine-Tuning on PQuAD

یک خط لوله کامل فاین‌تیونینگ مدل **ParsBERT** (سازگار با BERT فارسی) برای وظیفه **پرسش و پاسخ** روی دیتاست **PQuAD** با استفاده از HuggingFace `Trainer`.

**فهرست بخش‌ها:**
1. نصب کتابخانه‌ها و راه‌اندازی محیط (Google Drive)
2. ایمپورت‌ها و پیکربندی
3. بارگذاری دیتاست و توکنایزر
4. پیش‌پردازش داده (توکن‌ایزینگ پنجره متحرک)
5. مقداردهی مدل و آماده‌سازی آموزش
6. آموزش (چک‌پوینت دستی)، ارزیابی و ذخیره در درایو
7. بازیابی مدل ذخیره‌شده
8. ارزیابی نهایی روی داده اعتبارسنجی (معیار SQuAD v2)
9. انتشار روی HuggingFace Hub

> **نکته:** اجرای `trainer.train()` عمداً به‌صورت دستی (comment) نگه داشته شده است. برای فاین‌تیونینگ آن را فعال کنید؛ یا برای ارزیابی یک مدل از قبل آموزش‌دیده، مستقیم به بخش «بازیابی مدل ذخیره‌شده» بروید.


## 1. Setup - نصب کتابخانه‌ها

نصب نسخه‌های ثابت کتابخانه‌ها برای پایداری خروجی. این سلول فقط یک‌بار اجرا می‌شود.


In [ ]:
!pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!pip install accelerate==0.33.0 transformers==4.43.3 datasets==2.20.0 evaluate==0.4.2 tokenizers==0.19.1 peft==0.12.0 "hazm>=0.9.4" scikit-learn==1.5.1 numpy==1.26.4 pandas==2.2.2 tqdm==4.66.4 sentencepiece==0.2.0

## 2. Environment & Google Drive

اتصال به گوگل‌درایو و آماده‌سازی مسیر پکیج‌ها.


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive/')
# مسیر دلخواه داخل درایو برای ذخیره پکیج‌ها
package_path = "/content/drive/MyDrive/colab_env"
os.makedirs(package_path, exist_ok=True)
print("package path exists:", os.path.exists(package_path))


## 3. Imports & Configuration

ایمپورت کتابخانه‌ها و تعریف پیکربندی سراسری (طول پنجره، گام پنجره، نام مدل و مسیر ذخیره).


In [ ]:
import numpy as np
import pandas
import collections
import torch
import fsspec
import requests
import evaluate
import datasets

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from tqdm.auto import tqdm

print("numpy:", np.__version__)
print("pandas:", pandas.__version__)
print("evaluate:", evaluate.__version__)
print("Datasets:", datasets.__version__)
print("fsspec:", fsspec.__version__)
print("requests:", requests.__version__)
# print("CUDA available:", torch.cuda.is_available())
# if torch.cuda.is_available():
#     print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# پارامترهای پنجره متحرک برای متون طولانی
max_length = 384
doc_stride = 128

# مدل پایه پارس‌برت
model_checkpoint = "HooshvareLab/bert-fa-base-uncased"

# مسیر ذخیره مدل در درایو و نام مدل در هاب
save_path = "/content/drive/MyDrive/Persionmodel_parsbert"
hub_model_id = "Msoldier-ai/parsbert-qa"


## 4. بارگذاری دیتاست و توکنایزر

- بارگذاری دیتاست فارسی **PQuAD** (`Z-Jafari/PQuAD`)
- بارگذاری توکنایزر **ParsBERT**


In [ ]:
# ۱. بارگذاری دیتاست پرسش و پاسخ فارسی PQuAD
dataset = load_dataset("Z-Jafari/PQuAD")

# بارگذاری توکنایزر پارس‌برت
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

print(dataset)


#### مشاهده یک نمونه از دیتاست (Inspect a sample)


In [ ]:
sample = dataset['train'][0]

print(sample["context"])

print(sample["question"])

print(sample['answers'])

## 5. پیش‌پردازش داده - توکن‌ایزینگ پنجره متحرک

تابع `prepare_train_features` نمونه‌های خام را به فرمت موردنیاز مدل تبدیل می‌کند:

- توکن‌ایزینگ هم‌زمان سوال و متن با پنجره متحرک برای متون طولانی
- هم‌ترازی ایندکس `start`/`end` پاسخ با توکن‌ها
- قرار دادن برچسب پاسخ‌های بدون پاسخ روی توکن `[CLS]`


In [ ]:
def prepare_train_features(examples):
    # تمیزکاری فاصله‌های اضافی در سوال‌ها
    examples["question"] = [q.lstrip() for q in examples["question"]]

    # توکنایز کردن همزمان سوال و متن با در نظر گرفتن پنجره متحرک
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",  # فقط متن مرجع کوتاه شود، نه سوال
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    tokenized_examples["start_positions"] = []
    tokenized_examples["end_positions"] = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # تشخیص اینکه کدام بخش توکن‌ها مربوط به متن مرجع (Context) است
        sequence_ids = tokenized_examples.sequence_ids(i)
        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]

        # اگر سوال بدون پاسخ بود، برچسب‌ها روی توکن [CLS] قرار می‌گیرند
        if len(answers["answer_start"]) == 0:
            tokenized_examples["start_positions"].append(cls_index)
            tokenized_examples["end_positions"].append(cls_index)
        else:
            # شروع و پایان کاراکتری پاسخ
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # پیدا کردن محدوده توکن‌های متن در این پنجره
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1

            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # بررسی اینکه آیا پاسخ درون این برش از پنجره قرار گرفته یا خیر
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                tokenized_examples["start_positions"].append(cls_index)
                tokenized_examples["end_positions"].append(cls_index)
            else:
                # تبدیل ایندکس کاراکتر به ایندکس توکن شروع
                while token_start_index < len(offsets) and offsets[token_start_index][0] <= start_char:
                    token_start_index += 1
                tokenized_examples["start_positions"].append(token_start_index - 1)

                # تبدیل ایندکس کاراکتر به ایندکس توکن پایان
                while offsets[token_end_index][1] >= end_char:
                    token_end_index -= 1
                tokenized_examples["end_positions"].append(token_end_index + 1)

    return tokenized_examples


In [ ]:
# ۲. اعمال تابع توکنایزر روی کل دیتاست بخش آموزش
train_dataset = dataset["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing and aligning train dataset",
)


#### اعمال همان تابع روی داده‌های اعتبارسنجی


In [ ]:
# اعمال همان تابع توکنایزر روی داده‌های اعتبارسنجی
eval_dataset = dataset["validation"].map(
    prepare_train_features,
    batched=True,
    remove_columns=dataset["validation"].column_names,
    desc="Tokenizing validation dataset",
)

## 6. مقداردهی مدل و آماده‌سازی آموزش

مقداردهی مدل پایه با هد پرسش و پاسخ و تنظیم پارامترهای آموزش.

> **چک‌پوینت دستی:** اجرای `trainer.train()` عمداً comment شده است. برای شروع فاین‌تیونینگ، خط مربوطه را فعال کنید.


In [ ]:
# بررسی دسترسی به کارت گرافیک
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(f"Using device: {device}")

# بارگذاری مدل پایه با هد پرسش و پاسخ
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)
# model.to(device)


In [ ]:
# تنظیم پارامترهای آموزش
training_args = TrainingArguments(
    output_dir="./parsbert-pquad-qa",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=0,
    fp16=torch.cuda.is_available(),  # دقت ۱۶ بیت برای سرعت بیشتر و مصرف کمتر حافظه
    logging_steps=100,
    save_total_limit=2,  # نگه‌داری فقط ۲ چک‌پوینت آخر
    load_best_model_at_end=True,
    report_to="none",
)


In [ ]:
# تجمیع‌کننده داده‌ها برای دسته‌بندی تانسورها
data_collator = default_data_collator

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

# شروع فرآیند فاین‌تیونینگ
# trainer.train()


## 7. ارزیابی مدل فعلی و ذخیره در درایو

ارزیابی روی داده اعتبارسنجی با وزن‌های فعلی مدل (اگر بدون آموزش اجرا شود، روی مدل پایه است) و سپس ذخیره مدل و توکنایزر در گوگل‌درایو.


In [ ]:
trainer.evaluate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"Model and tokenizer saved successfully to: {save_path}")


## 8. بازیابی مدل ذخیره‌شده (چک‌پوینت دستی)

اگر این نوت‌بوک در جلسه‌ای جدید اجرا می‌شود، مدل و توکنایزر را از مسیر ذخیره‌شده بازیابی کنید. این بخش همچنین برای ارزیابی یک مدلِ از قبل آموزش‌دیده استفاده می‌شود.


In [ ]:
import torch
from transformers import AutoModelForQuestionAnswering, AutoTokenizer

drive.mount('/content/drive')

tokenizer = AutoTokenizer.from_pretrained(save_path)
model = AutoModelForQuestionAnswering.from_pretrained(save_path)


## 9. ارزیابی نهایی روی داده اعتبارسنجی

### 9.1 توکن‌ایزینگ مخصوص ارزیابی

تفاوت این بخش با آموزش: `example_id` اصلی حفظ می‌شود و افست‌های خارج از کانتکست `None` می‌شوند تا متن پاسخ به‌درستی بازسازی شود.


In [ ]:

# توکنایز کردن مخصوص ارزیابی
def prepare_validation_features(examples):
    examples["question"] = [q.lstrip() for q in examples["question"]]

    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=max_length,
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    tokenized_examples["example_id"] = []

    for i in range(len(tokenized_examples["input_ids"])):
        sample_index = sample_mapping[i]
        # حفظ شناسه نمونه اصلی برای بازسازی متن پاسخ
        tokenized_examples["example_id"].append(examples["id"][sample_index])

        # ماسک کردن افست‌های خارج از کانتکست
        sequence_ids = tokenized_examples.sequence_ids(i)
        offset = tokenized_examples["offset_mapping"][i]
        tokenized_examples["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offset)
        ]

    return tokenized_examples

val_dataset_eval = dataset["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=dataset["validation"].column_names,
    desc="Processing validation dataset for evaluation",
)

### 9.2 پیش‌بینی و پس‌پردازش خروجی‌ها

تابع `postprocess_qa_predictions` لاجیت‌های مدل را به پاسخ متنی تبدیل می‌کند (انتخاب بهترین بازه + تصمیم‌گیری درباره پاسخ‌نداشتن) و سپس پیش‌بینی روی دیتاست ارزیابی انجام می‌شود.

> یک `Trainer` جدید متصل به مدلِ بارگذاری‌شده ساخته می‌شود تا خروجی ارزیابی دقیقاً وزن‌های بازیابی‌شده را منعکس کند.


In [ ]:
def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=50):
    all_start_logits, all_end_logits = raw_predictions.predictions

    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature in enumerate(features):
        features_per_example[example_id_to_index[feature["example_id"]]].append(i)

    predictions = collections.OrderedDict()

    for example_index, example in enumerate(tqdm(examples, desc="Post-processing")):
        feature_indices = features_per_example[example_index]
        min_null_score = None
        valid_answers = []

        context = example["context"]

        for feature_index in feature_indices:
            start_logits = all_start_logits[feature_index]
            end_logits = all_end_logits[feature_index]
            offset_mapping = features[feature_index]["offset_mapping"]

            # امتیاز عدم پاسخ (توکن CLS)
            cls_index = features[feature_index]["input_ids"].index(tokenizer.cls_token_id)
            feature_null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or min_null_score < feature_null_score:
                min_null_score = feature_null_score

            # استخراج ۲۰ لاجیت برتر شروع و پایان
            start_indexes = np.argsort(start_logits)[-1 : -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1 : -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    # فیلتر کردن موارد نامعتبر
                    if (
                        start_index >= len(offset_mapping)
                        or end_index >= len(offset_mapping)
                        or offset_mapping[start_index] is None
                        or offset_mapping[end_index] is None
                        or end_index < start_index
                        or end_index - start_index + 1 > max_answer_length
                    ):
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    valid_answers.append({
                        "score": start_logits[start_index] + end_logits[end_index],
                        "text": context[start_char:end_char]
                    })

        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            best_answer = {"text": "", "score": 0.0}

        # اگر احتمال بدون پاسخ بودن بیشتر از بهترین پاسخ بود
        if min_null_score is not None and min_null_score > best_answer["score"]:
            predictions[example["id"]] = ""
        else:
            predictions[example["id"]] = best_answer["text"]

    return predictions


In [ ]:
# ساخت Trainer جدید متصل به مدل بارگذاری‌شده (برای اطمینان از ارزیابی وزن‌های بازیابی‌شده)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

raw_predictions = trainer.predict(val_dataset_eval)
final_predictions = postprocess_qa_predictions(
    dataset["validation"],
    val_dataset_eval,
    raw_predictions
)


### 9.3 محاسبه معیارها با SQuAD v2


In [ ]:
squad_metric = evaluate.load("squad_v2")

formatted_predictions = [{"id": k, "prediction_text": v, "no_answer_probability": 0.0} for k, v in final_predictions.items()]
formatted_references = [{"id": ex["id"], "answers": ex["answers"]} for ex in dataset["validation"]]

results = squad_metric.compute(predictions=formatted_predictions, references=formatted_references)

print("--- نتایج ارزیابی مدل روی PQuAD ---")
for key, value in results.items():
    print(f"{key}: {value:.2f}")

## 10. انتشار روی HuggingFace Hub

آپلود مستقیم مدل آموزش‌دیده و توکنایزر روی HuggingFace Hub.


In [ ]:
from huggingface_hub import login

# توکن دسترسی خود با دسترسی Write را وارد کنید
login()

# آپلود مستقیم آداپتورها و توکنایزر
hub_model_id = "Msoldier-ai/parsbert-qa"

model.push_to_hub(hub_model_id)
tokenizer.push_to_hub(hub_model_id)
print(f"Model pushed successfully: https://huggingface.co/{hub_model_id}")